**IMPORT THƯ VIỆN**

In [1]:
import pandas as pd
import numpy as np

**LOAD DỮ LIỆU**

In [16]:
# Đọc dữ liệu gốc (raw) - chưa qua bất kỳ xử lý làm sạch nào
customers_raw = pd.read_csv("olist_customers_dataset.csv")

# parse_dates: ép các cột ngày về kiểu datetime ngay khi đọc để tính toán được sau này
orders_raw = pd.read_csv("olist_orders_dataset.csv", parse_dates=[
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date"
])

order_items_raw = pd.read_csv("olist_order_items_dataset.csv",
                               parse_dates=["shipping_limit_date"])

payments_raw = pd.read_csv("olist_order_payments_dataset.csv")

reviews_raw = pd.read_csv("olist_order_reviews_dataset.csv", parse_dates=[
    "review_creation_date", "review_answer_timestamp"
])

products_raw = pd.read_csv("olist_products_dataset.csv")

sellers_raw = pd.read_csv("olist_sellers_dataset.csv")

# Bảng dịch tên category từ tiếng Bồ Đào Nha sang tiếng Anh
category_translation = pd.read_csv("product_category_name_translation.csv")

In [17]:
raw_shapes = {
    "customers": customers_raw.shape[0], "orders": orders_raw.shape[0],
    "order_items": order_items_raw.shape[0], "payments": payments_raw.shape[0],
    "reviews": reviews_raw.shape[0], "products": products_raw.shape[0],
    "sellers": sellers_raw.shape[0],
}
print("Số dòng dữ liệu GỐC (chưa xử lý):")
for k, v in raw_shapes.items():
    print(f"  {k}: {v}")

# Tạo bản làm việc (working copy) - luôn giữ *_raw nguyên vẹn để đối chiếu/kiểm tra lại
customers = customers_raw.copy()
orders = orders_raw.copy()
order_items = order_items_raw.copy()
payments = payments_raw.copy()
reviews = reviews_raw.copy()
products = products_raw.copy()
sellers = sellers_raw.copy()

Số dòng dữ liệu GỐC (chưa xử lý):
  customers: 99441
  orders: 99441
  order_items: 112650
  payments: 103886
  reviews: 99224
  products: 32951
  sellers: 3095


**DEDUPLICATE - KIỂM TRA & LOẠI TRÙNG LẶP**

In [28]:
dup_customers = customers.duplicated(subset=["customer_id"]).sum()
print(f"Số dòng trùng customer_id: {dup_customers}")
customers = customers.drop_duplicates(subset=["customer_id"])

# orders: order_id phải là khóa chính
dup_orders = orders.duplicated(subset=["order_id"]).sum()
print(f"Số dòng trùng order_id: {dup_orders}")
orders = orders.drop_duplicates(subset=["order_id"])

# order_items: khóa chính là (order_id, order_item_id)
dup_items = order_items.duplicated(subset=["order_id", "order_item_id"]).sum()
print(f"Số dòng trùng (order_id, order_item_id) trong order_items: {dup_items}")
order_items = order_items.drop_duplicates(subset=["order_id", "order_item_id"])

# payments: khóa chính là (order_id, payment_sequential)
dup_payments = payments.duplicated(subset=["order_id", "payment_sequential"]).sum()
print(f"Số dòng trùng (order_id, payment_sequential) trong payments: {dup_payments}")
payments = payments.drop_duplicates(subset=["order_id", "payment_sequential"])

# reviews: review_id có thể lặp lại nhiều dòng cho cùng 1 order_id trong bộ dữ
# liệu gốc Olist (do cấu trúc export) -> loại trùng hoàn toàn theo toàn bộ các cột, sau
# đó nếu 1 order_id có nhiều review_id khác nhau thì GIỮ review mới nhất theo
# review_creation_date (vì đó là đánh giá gần nhất/chính thức nhất của khách cho đơn đó)
before_reviews = len(reviews)
reviews = reviews.drop_duplicates()  # loại các dòng lặp y hệt nhau trước
reviews = (reviews.sort_values("review_creation_date")
           .drop_duplicates(subset=["order_id"], keep="last"))
print(f"Reviews: {before_reviews} -> {len(reviews)} dòng sau khi loại trùng "
      f"(giữ review mới nhất mỗi order_id)")

# products: product_id phải là khóa chính
dup_products = products.duplicated(subset=["product_id"]).sum()
print(f"Số dòng trùng product_id: {dup_products}")
products = products.drop_duplicates(subset=["product_id"])
dup_sellers = sellers.duplicated(subset=["seller_id"]).sum()
print(f"Số dòng trùng seller_id: {dup_sellers}")
sellers = sellers.drop_duplicates(subset=["seller_id"])

Số dòng trùng customer_id: 0
Số dòng trùng order_id: 0
Số dòng trùng (order_id, order_item_id) trong order_items: 0
Số dòng trùng (order_id, payment_sequential) trong payments: 0
Reviews: 98673 -> 98673 dòng sau khi loại trùng (giữ review mới nhất mỗi order_id)
Số dòng trùng product_id: 0
Số dòng trùng seller_id: 0


**NULL HANDLING - XỬ LÝ GIÁ TRỊ THIẾU**

In [29]:
null_orders = orders.isnull().sum()
print("Null trong orders (theo cột):\n", null_orders[null_orders > 0])

invalid_delivered_null = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isnull())
]
print(f"\nSố đơn status='delivered' nhưng thiếu ngày giao thực tế "
      f"(mâu thuẫn logic): {len(invalid_delivered_null)}")
orders = orders.drop(index=invalid_delivered_null.index)

# reviews: review_comment_title / review_comment_message là text tự do, khách
# thường để trống -> đây là null HỢP LỆ (không phải thiếu dữ liệu lỗi), điền chuỗi rỗng
# thay vì drop dòng để không mất thông tin review_score (vốn vẫn đầy đủ)
null_reviews = reviews.isnull().sum()
print("\nNull trong reviews (theo cột):\n", null_reviews[null_reviews > 0])
reviews["review_comment_title"] = reviews["review_comment_title"].fillna("")
reviews["review_comment_message"] = reviews["review_comment_message"].fillna("")

# products: product_category_name và các thông tin kích thước/cân nặng
null_products = products.isnull().sum()
print("\nNull trong products (theo cột):\n", null_products[null_products > 0])
products["product_category_name"] = products["product_category_name"].fillna("unknown")
# Với kích thước/cân nặng: dùng median theo category để giữ tính hợp lý hơn median toàn
# cục (một sản phẩm "unknown" không nên nhận median của cả sản phẩm nặng lẫn nhẹ)
for col in ["product_weight_g", "product_length_cm", "product_height_cm",
            "product_width_cm"]:
    products[col] = products.groupby("product_category_name")[col].transform(
        lambda x: x.fillna(x.median())
    )
    products[col] = products[col].fillna(products[col].median())  # phòng trường hợp
    # cả nhóm category đó toàn null

# order_items / payments: kiểm tra null (thường rất ít/không có ở 2 bảng này)
print("\nNull trong order_items:\n", order_items.isnull().sum().loc[lambda s: s > 0])
print("\nNull trong payments:\n", payments.isnull().sum().loc[lambda s: s > 0])
print("\nNull trong sellers:\n", sellers.isnull().sum().loc[lambda s: s > 0])

Null trong orders (theo cột):
 order_approved_at                 160
order_delivered_carrier_date     1781
order_delivered_customer_date    2957
delivery_time_days               2957
dtype: int64

Số đơn status='delivered' nhưng thiếu ngày giao thực tế (mâu thuẫn logic): 0

Null trong reviews (theo cột):
 Series([], dtype: int64)

Null trong products (theo cột):
 product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
dtype: int64

Null trong order_items:
 Series([], dtype: int64)

Null trong payments:
 Series([], dtype: int64)

Null trong sellers:
 Series([], dtype: int64)


**INVALID HANDLING - XỬ LÝ GIÁ TRỊ KHÔNG HỢP LỆ / PHI LOGIC**

In [35]:
# order_items: price và freight_value phải > 0 (không có giao dịch giá 0 hoặc âm)
invalid_price = order_items[
    (order_items["price"] <= 0) | (order_items["freight_value"] < 0)
]
print(f"Số dòng order_items có price<=0 hoặc freight_value<0: {len(invalid_price)}")
order_items = order_items.drop(index=invalid_price.index)

# reviews: review_score phải trong khoảng hợp lệ [1, 5]
invalid_score = reviews[~reviews["review_score"].between(1, 5)]
print(f"Số dòng reviews có review_score ngoài khoảng [1,5]: {len(invalid_score)}")
reviews = reviews.drop(index=invalid_score.index)

# orders: kiểm tra logic thời gian - ngày giao không thể SỚM HƠN ngày mua, ngày
# duyệt đơn không thể sớm hơn ngày mua
invalid_dates = orders[
    (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]) |
    (orders["order_approved_at"] < orders["order_purchase_timestamp"])
]
print(f"Số đơn có ngày giao/duyệt sớm hơn ngày mua (phi logic): {len(invalid_dates)}")
orders = orders.drop(index=invalid_dates.index)

# payments: payment_installments phải >= 1 (số kỳ trả góp tối thiểu là 1 lần)
invalid_installments = payments[payments["payment_installments"] < 1]
print(f"Số dòng payments có payment_installments < 1: {len(invalid_installments)}")
payments = payments.drop(index=invalid_installments.index)

# Khóa ngoại mồ côi (orphan foreign key): order_items tham chiếu đến order_id
# không tồn tại trong orders (có thể do orders đã bị loại ở bước trước, hoặc lỗi gốc)
orphan_items = order_items[~order_items["order_id"].isin(orders["order_id"])]
print(f"Số dòng order_items tham chiếu order_id không tồn tại trong orders: "
      f"{len(orphan_items)}")
order_items = order_items[order_items["order_id"].isin(orders["order_id"])]

# Tương tự, kiểm tra order_items tham chiếu đến product_id không tồn tại trong products
orphan_items_product = order_items[~order_items["product_id"].isin(products["product_id"])]
print(f"Số dòng order_items tham chiếu product_id không tồn tại trong products: "
      f"{len(orphan_items_product)}")
order_items = order_items[order_items["product_id"].isin(products["product_id"])]

# Kiểm tra payments tham chiếu đến order_id không tồn tại trong orders
orphan_payments = payments[~payments["order_id"].isin(orders["order_id"])]
print(f"Số dòng payments tham chiếu order_id không tồn tại trong orders: "
      f"{len(orphan_payments)}")
payments = payments[payments["order_id"].isin(orders["order_id"])]

# Kiểm tra order_items tham chiếu đến seller_id không tồn tại trong sellers
orphan_items_seller = order_items[~order_items["seller_id"].isin(sellers["seller_id"])]
print(f"Số dòng order_items tham chiếu seller_id không tồn tại trong sellers: "
      f"{len(orphan_items_seller)}")
order_items = order_items[order_items["seller_id"].isin(sellers["seller_id"])]

# Chiều ngược lại: seller không có đơn hàng nào trong order_items sau khi lọc - KHÔNG
# xoá khỏi bảng sellers, vì đây là seller mới/ít đơn (dữ liệu hợp lệ), không phải lỗi
# giống trường hợp khóa ngoại mồ côi ở trên (nơi con trỏ trỏ đến cha không tồn tại)
n_seller_no_order = (~sellers["seller_id"].isin(order_items["seller_id"])).sum()
print(f"Số seller không có đơn hàng nào trong order_items (giữ nguyên, không xoá): "
      f"{n_seller_no_order}")

Số dòng order_items có price<=0 hoặc freight_value<0: 0
Số dòng reviews có review_score ngoài khoảng [1,5]: 0
Số đơn có ngày giao/duyệt sớm hơn ngày mua (phi logic): 0
Số dòng payments có payment_installments < 1: 0
Số dòng order_items tham chiếu order_id không tồn tại trong orders: 0
Số dòng order_items tham chiếu product_id không tồn tại trong products: 0
Số dòng payments tham chiếu order_id không tồn tại trong orders: 0
Số dòng order_items tham chiếu seller_id không tồn tại trong sellers: 0
Số seller không có đơn hàng nào trong order_items (giữ nguyên, không xoá): 211


**XOÁ OUTLIER - DÙNG PHƯƠNG PHÁP IQR (Interquartile Range)**

In [21]:
def remove_outliers_iqr(df, column, k=3.0):
    """Xoá outlier theo phương pháp IQR với hệ số k (mặc định 3.0 - bảo thủ hơn 1.5
    thường dùng, vì mục tiêu là loại bỏ giá trị BẤT THƯỜNG chứ không phải giá trị cao
    nhưng vẫn hợp lý về mặt kinh doanh)."""
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - k * iqr
    upper_bound = q3 + k * iqr
    mask = df[column].between(lower_bound, upper_bound)
    n_removed = (~mask).sum()
    print(f"  Cột '{column}': loại {n_removed} outlier "
          f"(ngưỡng hợp lệ: [{lower_bound:.2f}, {upper_bound:.2f}])")
    return df[mask]


print("Xoá outlier trong order_items:")
before_items = len(order_items)
order_items = remove_outliers_iqr(order_items, "price", k=3.0)
order_items = remove_outliers_iqr(order_items, "freight_value", k=3.0)
print(f"  order_items: {before_items} -> {len(order_items)} dòng")

# Outlier về thời gian giao hàng thực tế (số ngày từ lúc mua đến lúc nhận hàng) - đơn
# giao trong 0 ngày (lỗi timestamp) hoặc giao sau hàng trăm ngày (bất thường so với vận
# hành thông thường) đều cần loại trước khi đưa vào tính feature delivery_delay
orders["delivery_time_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

print("\nXoá outlier trong orders (delivery_time_days):")
before_orders = len(orders)
# Loại thêm trường hợp phi logic: thời gian giao <= 0 ngày (không thể giao trong hoặc
# trước ngày đặt hàng đối với thương mại điện tử vật lý). Đơn CHƯA giao (delivery_time
# null vì chưa có order_delivered_customer_date) vẫn được GIỮ LẠI ở bảng orders - chỉ
# loại khỏi việc tính outlier, không loại khỏi dữ liệu.
orders = orders[(orders["delivery_time_days"].isnull()) |
                 (orders["delivery_time_days"] > 0)]

# Tách 2 nhóm: có delivery_time (để lọc outlier) và không có (giữ nguyên, không lọc)
orders_with_time = orders[orders["delivery_time_days"].notnull()]
orders_without_time = orders[orders["delivery_time_days"].isnull()]

orders_with_time_clean = remove_outliers_iqr(orders_with_time, "delivery_time_days",
                                              k=3.0)
orders = pd.concat([orders_with_time_clean, orders_without_time], ignore_index=False)
print(f"  orders: {before_orders} -> {len(orders)} dòng")

Xoá outlier trong order_items:
  Cột 'price': loại 4074 outlier (ngưỡng hợp lệ: [-245.10, 419.90])
  Cột 'freight_value': loại 5096 outlier (ngưỡng hợp lệ: [-9.64, 42.93])
  order_items: 112642 -> 103472 dòng

Xoá outlier trong orders (delivery_time_days):
  Cột 'delivery_time_days': loại 1320 outlier (ngưỡng hợp lệ: [-21.00, 42.00])
  orders: 99433 -> 98100 dòng


**TỔNG KẾT - SO SÁNH SỐ DÒNG TRƯỚC/SAU LÀM SẠCH**

In [36]:
# Đếm số dòng còn lại của từng bảng SAU khi đã làm sạch (deduplicate, null handling,
# invalid handling, xoá outlier)
clean_shapes = {
    "customers": len(customers), "orders": len(orders),
    "order_items": len(order_items), "payments": len(payments),
    "reviews": len(reviews), "products": len(products),
    "sellers": len(sellers),
}

print("=== BẢNG SO SÁNH TRƯỚC / SAU LÀM SẠCH ===")
# Gộp số dòng gốc (raw_shapes) và số dòng sau làm sạch (clean_shapes) vào 1 DataFrame
# để so sánh trực quan theo từng bảng
summary_df = pd.DataFrame({
    "raw": raw_shapes,
    "clean": clean_shapes,
})

# Tính số dòng bị loại và tỷ lệ % bị loại của mỗi bảng - đây là con số cần trích vào
# báo cáo Word (mục Data Preparation) để chứng minh việc làm sạch có tác động cụ thể
summary_df["removed"] = summary_df["raw"] - summary_df["clean"]
summary_df["removed_pct"] = (summary_df["removed"] / summary_df["raw"] * 100).round(2)
print(summary_df)

=== BẢNG SO SÁNH TRƯỚC / SAU LÀM SẠCH ===
                raw   clean  removed  removed_pct
customers     99441   99441        0         0.00
orders        99441   98100     1341         1.35
order_items  112650  102219    10431         9.26
payments     103886  102460     1426         1.37
reviews       99224   98673      551         0.56
products      32951   32951        0         0.00
sellers        3095    3095        0         0.00


**Xuất dữ liệu**

In [34]:
customers.to_csv("clean_customers.csv", index=False)
orders.to_csv("clean_orders.csv", index=False)
order_items.to_csv("clean_order_items.csv", index=False)
payments.to_csv("clean_payments.csv", index=False)
reviews.to_csv("clean_reviews.csv", index=False)
products.to_csv("clean_products.csv", index=False)
sellers.to_csv("clean_sellers.csv", index=False)
print("Đã xuất 7 file clean_*.csv - dùng làm đầu vào cho bước Feature Engineering")

Đã xuất 7 file clean_*.csv - dùng làm đầu vào cho bước Feature Engineering
